# Time Series Discovery — Agentic Systems 2.1.9

This notebook teaches the core grammar with a frozen FAOSTAT JSON snapshot. It targets SDK 2.1.9. Candidate execution is pending; the historical Native and OpenAI Agents runs in Recorded_Outputs.ipynb used SDK 2.1.8 and retain their original identities.

Run the local profile first. Bedrock is disabled by default. To test a live profile in ADA, select the framework, supply the model and Boto3 session, review the explicit persistent request cap, and bind PYPI_WHEEL to the exact candidate artifact. Each campaign must retain its ledger; do not reset it to bypass a cap.

The 2.1.9 candidate uses its own versioned root and campaign. This is separate from the preserved 2.1.8 campaign; no previous request count is deleted. Model/provider access and candidate acceptance are NOT RUN until supported by the new artifact's evidence.

Explicit Agent contracts and policies are outside this lesson. They remain advanced API features, with broader validation planned for 2.1.10.


In [ ]:
# Set True only when installing into this kernel. No shell/PATH ambiguity.
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        'agentic-systems[bedrock,openai-agents]==2.1.9',
        'matplotlib>=3.8', 'nbformat>=5.10', 'nbclient>=0.10', 'ipykernel>=6'])
    print('Installation complete. Restart kernel if the SDK was previously imported.')

In [ ]:
import os, sys, json, math, hashlib, inspect, uuid
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
from statistics import mean
from typing import Literal
from pydantic import BaseModel, Field, field_validator
from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt
import agentic_systems as toolkit

assert version('agentic-systems') == '2.1.9', 'Install the pinned SDK and restart kernel.'
RUN_BEDROCK = False
LIVE_FRAMEWORKS = ("openai-agents",)  # Only pending profile. Or ("native",) / ("native", "openai-agents").
# No OpenAI API key: Bedrock remains the provider.
BEDROCK_MODEL_ID = 'qwen.qwen3-32b-v1:0'  # Exact Qwen3 32B ID/profile available in ADA.
AWS_REGION = 'us-east-1'
# Reuse `session` if already defined in ADA, or assign a Boto3/SageMaker session here.
ADA_SESSION = globals().get('session', None)
if ADA_SESSION is None:
    ADA_SESSION = globals().get('sagemaker_session', None)
# Optional: ADA_SESSION = sagemaker_session.boto_session

ROOT = Path.cwd() / 'time_series_discovery_219'
ROOT.mkdir(exist_ok=True)
CAMPAIGN_ID = 'ada-219-core'  # Keep stable across reruns; do not reset to evade the cap.
MAX_BEDROCK_HTTP_ATTEMPTS = 100  # Explicit cumulative cap: previous 60 + up to 40 additional sends.
BUDGET_CAP_INCREASE_FROM = 60   # Only this prior cap may be extended; recorded in the same ledger.
ALLOW_NEW_CAMPAIGN = True     # Existing ADA campaign expected. True ONLY for a first-ever run.
MAX_OUTPUT_TOKENS = 1024
OUTPUT = ROOT / 'runs' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ_%f')
OUTPUT.mkdir(parents=True, exist_ok=False)
VERSIONS = {x: version(x) for x in ['agentic-systems','langgraph','pydantic','boto3','openai-agents']}
print(VERSIONS)
print('Output directory:', OUTPUT)
assert LIVE_FRAMEWORKS and len(set(LIVE_FRAMEWORKS)) == len(LIVE_FRAMEWORKS)
assert set(LIVE_FRAMEWORKS) <= {"native", "openai-agents"}
print('Local profile: python-runtime/native; Bedrock enabled:', RUN_BEDROCK)
print('Selected live frameworks:', LIVE_FRAMEWORKS)
print('Campaign ledger:', ROOT / (CAMPAIGN_ID + '.sqlite'))
print('Explicit cumulative HTTP cap:', MAX_BEDROCK_HTTP_ATTEMPTS)

In [ ]:
# Frozen source snapshot. This cell creates exactly the same JSON in standalone use.
SNAPSHOT_TEXT = '{\n  "schema": "time-series-snapshot.v1",\n  "source": {\n    "publisher": "FAO",\n    "dataset": "FAOSTAT Production Indices (QI)",\n    "url": "https://bulks-faostat.fao.org/production/Production_Indices_E_All_Data_(Normalized).zip",\n    "metadata_url": "https://www.fao.org/faostat/en/#data/QI/metadata",\n    "terms_url": "https://www.fao.org/contact-us/terms/db-terms-of-use/en/",\n    "retrieved_at_utc": "2026-11-11T23:08:36.601107+00:00",\n    "archive_sha256": "04c58034493d91f02e2495570d3864212ac870ec7f829a5f22577d143a6a8a72",\n    "member": "Production_Indices_E_All_Data_(Normalized).csv",\n    "selection": {\n      "Area": "Mexico",\n      "Item": "Agriculture",\n      "Element Code": "432"\n    },\n    "transformation": "Filter exact source identifiers; cast Year to int and Value to float; retain Flag; sort by Year. No imputation, smoothing, rescaling or synthetic observations."\n  },\n  "series": {\n    "area": "Mexico",\n    "area_code": "138",\n    "item": "Agriculture",\n    "item_code": "2051",\n    "element": "Gross Production Index Number (2014-2016 = 100)",\n    "element_code": "432",\n    "source_unit": "",\n    "display_unit": "index points (2014-2016 = 100)",\n    "frequency": "annual"\n  },\n  "flags": [\n    {\n      "Flag": "E",\n      " Description": "Estimated value"\n    }\n  ],\n  "observations": [\n    {\n      "year": 1961,\n      "value": 20.49,\n      "flag": "E"\n    },\n    {\n      "year": 1962,\n      "value": 21.83,\n      "flag": "E"\n    },\n    {\n      "year": 1963,\n      "value": 22.95,\n      "flag": "E"\n    },\n    {\n      "year": 1964,\n      "value": 25.11,\n      "flag": "E"\n    },\n    {\n      "year": 1965,\n      "value": 26.81,\n      "flag": "E"\n    },\n    {\n      "year": 1966,\n      "value": 28.44,\n      "flag": "E"\n    },\n    {\n      "year": 1967,\n      "value": 28.92,\n      "flag": "E"\n    },\n    {\n      "year": 1968,\n      "value": 29.75,\n      "flag": "E"\n    },\n    {\n      "year": 1969,\n      "value": 29.17,\n      "flag": "E"\n    },\n    {\n      "year": 1970,\n      "value": 30.55,\n      "flag": "E"\n    },\n    {\n      "year": 1971,\n      "value": 31.88,\n      "flag": "E"\n    },\n    {\n      "year": 1972,\n      "value": 33.38,\n      "flag": "E"\n    },\n    {\n      "year": 1973,\n      "value": 34.7,\n      "flag": "E"\n    },\n    {\n      "year": 1974,\n      "value": 35.84,\n      "flag": "E"\n    },\n    {\n      "year": 1975,\n      "value": 36.94,\n      "flag": "E"\n    },\n    {\n      "year": 1976,\n      "value": 37.16,\n      "flag": "E"\n    },\n    {\n      "year": 1977,\n      "value": 40.97,\n      "flag": "E"\n    },\n    {\n      "year": 1978,\n      "value": 44.08,\n      "flag": "E"\n    },\n    {\n      "year": 1979,\n      "value": 42.92,\n      "flag": "E"\n    },\n    {\n      "year": 1980,\n      "value": 46.86,\n      "flag": "E"\n    },\n    {\n      "year": 1981,\n      "value": 50.14,\n      "flag": "E"\n    },\n    {\n      "year": 1982,\n      "value": 49.15,\n      "flag": "E"\n    },\n    {\n      "year": 1983,\n      "value": 51.29,\n      "flag": "E"\n    },\n    {\n      "year": 1984,\n      "value": 51.8,\n      "flag": "E"\n    },\n    {\n      "year": 1985,\n      "value": 53.59,\n      "flag": "E"\n    },\n    {\n      "year": 1986,\n      "value": 55.0,\n      "flag": "E"\n    },\n    {\n      "year": 1987,\n      "value": 55.45,\n      "flag": "E"\n    },\n    {\n      "year": 1988,\n      "value": 54.47,\n      "flag": "E"\n    },\n    {\n      "year": 1989,\n      "value": 51.86,\n      "flag": "E"\n    },\n    {\n      "year": 1990,\n      "value": 54.9,\n      "flag": "E"\n    },\n    {\n      "year": 1991,\n      "value": 56.06,\n      "flag": "E"\n    },\n    {\n      "year": 1992,\n      "value": 56.62,\n      "flag": "E"\n    },\n    {\n      "year": 1993,\n      "value": 58.2,\n      "flag": "E"\n    },\n    {\n      "year": 1994,\n      "value": 60.43,\n      "flag": "E"\n    },\n    {\n      "year": 1995,\n      "value": 63.24,\n      "flag": "E"\n    },\n    {\n      "year": 1996,\n      "value": 65.25,\n      "flag": "E"\n    },\n    {\n      "year": 1997,\n      "value": 66.38,\n      "flag": "E"\n    },\n    {\n      "year": 1998,\n      "value": 69.49,\n      "flag": "E"\n    },\n    {\n      "year": 1999,\n      "value": 71.41,\n      "flag": "E"\n    },\n    {\n      "year": 2000,\n      "value": 72.19,\n      "flag": "E"\n    },\n    {\n      "year": 2001,\n      "value": 76.19,\n      "flag": "E"\n    },\n    {\n      "year": 2002,\n      "value": 75.9,\n      "flag": "E"\n    },\n    {\n      "year": 2003,\n      "value": 78.46,\n      "flag": "E"\n    },\n    {\n      "year": 2004,\n      "value": 80.72,\n      "flag": "E"\n    },\n    {\n      "year": 2005,\n      "value": 80.09,\n      "flag": "E"\n    },\n    {\n      "year": 2006,\n      "value": 83.88,\n      "flag": "E"\n    },\n    {\n      "year": 2007,\n      "value": 85.97,\n      "flag": "E"\n    },\n    {\n      "year": 2008,\n      "value": 88.18,\n      "flag": "E"\n    },\n    {\n      "year": 2009,\n      "value": 85.98,\n      "flag": "E"\n    },\n    {\n      "year": 2010,\n      "value": 88.71,\n      "flag": "E"\n    },\n    {\n      "year": 2011,\n      "value": 87.69,\n      "flag": "E"\n    },\n    {\n      "year": 2012,\n      "value": 92.28,\n      "flag": "E"\n    },\n    {\n      "year": 2013,\n      "value": 94.53,\n      "flag": "E"\n    },\n    {\n      "year": 2014,\n      "value": 97.84,\n      "flag": "E"\n    },\n    {\n      "year": 2015,\n      "value": 98.35,\n      "flag": "E"\n    },\n    {\n      "year": 2016,\n      "value": 103.82,\n      "flag": "E"\n    },\n    {\n      "year": 2017,\n      "value": 108.38,\n      "flag": "E"\n    },\n    {\n      "year": 2018,\n      "value": 111.07,\n      "flag": "E"\n    },\n    {\n      "year": 2019,\n      "value": 112.12,\n      "flag": "E"\n    },\n    {\n      "year": 2020,\n      "value": 112.81,\n      "flag": "E"\n    },\n    {\n      "year": 2021,\n      "value": 114.42,\n      "flag": "E"\n    },\n    {\n      "year": 2022,\n      "value": 117.13,\n      "flag": "E"\n    },\n    {\n      "year": 2023,\n      "value": 120.41,\n      "flag": "E"\n    },\n    {\n      "year": 2024,\n      "value": 119.15,\n      "flag": "E"\n    }\n  ]\n}\n'
DATA_SHA256 = '59109a8fc4f18b0bf73ab9a19df2357800a303debaa23ceb2f505363f271fab2'
DATA_PATH = ROOT / 'data' / 'series.json'
DATA_PATH.parent.mkdir(exist_ok=True)
if not DATA_PATH.exists():
    DATA_PATH.write_text(SNAPSHOT_TEXT, encoding='utf-8')
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == DATA_SHA256, 'Snapshot changed; inspect before running.'
snapshot = json.loads(DATA_PATH.read_text())
frame = pd.DataFrame(snapshot['observations'])
assert frame.year.is_unique and frame.year.is_monotonic_increasing
assert frame.value.notna().all() and all(math.isfinite(v) for v in frame.value)
assert frame.year.tolist() == list(range(1961, 2025))
display(pd.DataFrame([snapshot['series']]))
display(frame.head(), frame.tail(), pd.DataFrame(snapshot['flags']))
print('Source:', snapshot['source']['url'])
print('Snapshot SHA256:', DATA_SHA256)
# Fixed windows selected by year, before evaluating whether they exceed any policy.
EARLIER = (2005, 2009)
LATER = (2010, 2014)

def make_record(first, last):
    selected = frame[frame.year.between(first, last)]
    assert len(selected) == last-first+1 and len(selected) % 2 == 0
    return {'question': 'Compare the earlier and later observations and show supporting measurements.',
            'years': selected.year.astype(int).tolist(), 'values': selected.value.tolist()}

sample = make_record(2005, 2014)
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(frame.year, frame.value, color='#175b8f', label='FAOSTAT observations')
ax.axvspan(*EARLIER, color='#6cc4b2', alpha=.3, label='Earlier window')
ax.axvspan(*LATER, color='#f2b35f', alpha=.3, label='Later window')
ax.set(xlabel='Year', ylabel=snapshot['series']['display_unit'], title='Mexico — Agriculture production index')
ax.legend(); fig.tight_layout(); plt.show()

## Execution support

Exports are deliberately separate from the teaching declarations. Results are written **before** acceptance assertions, including failures. The manifest contains versions, source hashes and profile identity, never the session or credentials. No AWS resource is created, no S3 bucket is needed, and no STS probe is required.

Before live acceptance, bind the wheel identity below to the exact 2.1.9 candidate filename and SHA256. Installation version and the imported SDK file hash are recorded separately; a version string alone does not verify a wheel.

In [ ]:
PYPI_WHEEL = {'status': 'PENDING: bind to the exact 2.1.9 candidate wheel before live acceptance'}
RESULTS = {}

def save_result(name, result):
    RESULTS[name] = result
    (OUTPUT / (name+'.evidence.json')).write_text(json.dumps(result.evidence(), indent=2))
    (OUTPUT / (name+'.lineage.json')).write_text(json.dumps(result.lineage().to_dict(), indent=2))
    (OUTPUT / (name+'.trace.json')).write_text(json.dumps(result.trace(), indent=2))
    inv = result.check_invariants()
    (OUTPUT / (name+'.invariants.json')).write_text(json.dumps(inv.to_dict(), indent=2))
    print(name, 'status=', result.status, 'invariants=', inv.ok)
    return result

def require_ok(result):
    assert result.ok, result.human_results()
    assert result.check_invariants().ok, result.check_invariants().to_dict()
    return result

def tool_events(result):
    return [event for run in result.walk() for event in run.tool_events]

def business_output(event):
    output = event.output
    # Known SDK/provider envelope, not arbitrary response repair.
    return output['data'] if isinstance(output, dict) and 'data' in output else output

manifest = {'sdk_target':'2.1.9', 'next_development_target':'2.1.9', 'versions':VERSIONS,
            'python':sys.version, 'data_sha256':DATA_SHA256, 'pypi_wheel':PYPI_WHEEL,
            'imported_sdk_init_sha256':hashlib.sha256(Path(toolkit.__file__).read_bytes()).hexdigest(),
            'live_requested':RUN_BEDROCK, 'bedrock_model':BEDROCK_MODEL_ID if RUN_BEDROCK else None,
            'region':AWS_REGION if RUN_BEDROCK else None, 'session_exported':False}
(OUTPUT/'manifest.json').write_text(json.dumps(manifest, indent=2))

def write_run_bundle():
    # Export even after a failed live stage. Exclude checkpoints and put bundle outside the run.
    from IPython.display import FileLink
    bundle = {'run_directory': OUTPUT.name,
              'bundled_at_utc': datetime.now(timezone.utc).isoformat(),
              'files': {}, 'read_errors': {}}
    for path in sorted(OUTPUT.rglob('*.json')):
        if '.ipynb_checkpoints' in path.parts:
            continue
        relative = path.relative_to(OUTPUT).as_posix()
        try:
            bundle['files'][relative] = json.loads(path.read_text(encoding='utf-8'))
        except (OSError, ValueError) as exc:
            bundle['read_errors'][relative] = str(exc)
    destination = OUTPUT.parent / (OUTPUT.name + '_bundle.json')
    destination.write_text(json.dumps(bundle, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Evidence bundle:', destination)
    display(FileLink(str(destination.relative_to(Path.cwd()))))
    return destination


## 1. A comparison that can be checked

**user:** Did the level of this series change? Compare the observations and show what supports the finding.

**agentic-systems:** The next cells compute and display the observed response. No model result is pre-recorded or simulated.

**internals:** A graph-backed actuator calculates the means of two fixed five-year windows. The displayed answer comes from the recorded Tool event.

**grammar:** Tool → Graph → Agent actuator → RunResult

**route:** question + observations → comparison actuator → compare_windows → measurement

In [ ]:
class Investigation(BaseModel):
    question: str
    years: list[int] = Field(min_length=4)
    values: list[float] = Field(min_length=4)
    profile: dict = Field(default_factory=dict)
    measurement: dict = Field(default_factory=dict)
    review: dict = Field(default_factory=dict)
    selected_tool: str | None = None
    analysis_checked: bool = False
    report: str | None = None

    @field_validator('values')
    @classmethod
    def finite_values(cls, values):
        if not all(math.isfinite(x) for x in values):
            raise ValueError('Finite observations required')
        return values

@toolkit.tool
def compare_windows(values: list[float]) -> dict:
    """Compare equal earlier/later windows using their arithmetic means."""
    if len(values) < 4 or len(values) % 2 or not all(math.isfinite(x) for x in values):
        raise ValueError('At least four finite values and equal-sized windows required')
    n = len(values)//2
    earlier, later = mean(values[:n]), mean(values[n:])
    return {'measurement': {'kind':'compare', 'value':later-earlier,
            'details': {'earlier_mean':earlier, 'later_mean':later, 'window_size':n}}}

comparison_graph = toolkit.Graph(Investigation)
comparison_graph.add_node('compare', compare_windows)
comparison_graph.add_edge(toolkit.START, 'compare')
comparison_graph.add_edge('compare', toolkit.END)
comparison_actuator = toolkit.Agent('comparison_actuator', graph=comparison_graph)
comparison = require_ok(save_result('01_comparison', await comparison_actuator.arun(sample)))
m = comparison.data['measurement']
display(Markdown(f"**agentic-systems:** Earlier mean: **{m['details']['earlier_mean']:.3f}**; later mean: **{m['details']['later_mean']:.3f}**; difference: **{m['value']:+.3f} index points**. This is a descriptive period comparison, not a structural-break or causal conclusion."))
assert any(e.name == 'compare_windows' and e.ok for e in tool_events(comparison))

## 2. Actions selected by an actor

**user:** Describe the observations instead of comparing the windows.

**agentic-systems:** The next cells compute and display the observed response. No model result is pre-recorded or simulated.

**internals:** Model actors can choose among profile, comparison and endpoint-slope Tools. Python executes an explicitly named single Tool; it does not interpret natural language.

**grammar:** Skill → Runtime → Framework → Agent actor → recorded Tool execution

**route:** request → actor → chosen Tool → recorded Tool output

In [ ]:
@toolkit.tool
def profile_series(values: list[float]) -> dict:
    """Describe finite observations without comparing periods."""
    if not values or not all(math.isfinite(x) for x in values):
        raise ValueError('Finite observations required')
    return {'profile': {'count':len(values), 'average':mean(values), 'minimum':min(values), 'maximum':max(values)}}

@toolkit.tool
def measure_trend(values: list[float]) -> dict:
    """Compute only the endpoint slope per observation interval, not a fitted trend."""
    if len(values) < 2:
        raise ValueError('At least two observations required')
    return {'measurement':{'kind':'endpoint_slope', 'value':(values[-1]-values[0])/(len(values)-1),
                           'details':{'intervals':len(values)-1}}}

quality_skill = toolkit.skill(name='data_quality', tools=(profile_series,), instructions='Inspect the observations.')
exploration_skill = toolkit.skill(name='series_exploration', tools=(profile_series, compare_windows, measure_trend),
    instructions='Use Tools for arithmetic. Choose a Tool appropriate to the question. Preserve all observations. No causal claims.')
local_runtime = toolkit.runtime(provider='python-runtime')
local_framework = toolkit.framework('native')
describer = toolkit.Agent('description_actor', runtime=local_runtime, framework=local_framework, skills=(quality_skill,))
description = require_ok(save_result('02_description_python', await describer.arun(
    {'tool':'profile_series', 'input':{'values':sample['values']}})))
print(description.human_results())
assert any(e.name=='profile_series' and e.ok and e.input['values']==sample['values'] for e in tool_events(description))

## 3. Coordinate an investigation

**user:** Compare the periods and check the measurement before forwarding a finding.

**agentic-systems:** The next cells compute and display the observed response. No model result is pre-recorded or simulated.

**internals:** Profiler is an actuator, Investigator an actor, Analyst an actuator. The callback checks exact Tool arguments and retains nested actor execution in the parent evidence.

**grammar:** System → coordination Graph → Context → state channels

**route:** Discovery → Profiler → Investigator → Analyst

In [ ]:
@toolkit.tool
def validate_measurement(values: list[float], measurement: dict, years: list[int]) -> dict:
    # This Tool is used by a graph actuator, never exposed as an open-dict model schema.
    if len(values) != len(years) or len(set(years)) != len(years) or years != sorted(years):
        raise ValueError('Observation/year alignment is invalid')
    n=len(values)//2
    if measurement.get('kind') != 'compare' or len(values)%2:
        raise ValueError('Expected a period comparison')
    details=measurement['details']
    expected={'earlier_mean':mean(values[:n]), 'later_mean':mean(values[n:])}
    if not all(math.isclose(details[k], v, rel_tol=1e-10, abs_tol=1e-10) for k,v in expected.items()):
        raise ValueError('Recorded means do not match observations')
    if not math.isclose(measurement['value'], expected['later_mean']-expected['earlier_mean'], rel_tol=1e-10, abs_tol=1e-10):
        raise ValueError('Recorded difference is inconsistent')
    return {'analysis_checked':True}

# A declared business screening rule, not a significance threshold; not fitted to obtain a candidate.
context = toolkit.Context({'dataset':'FAOSTAT-QI-Mexico-Agriculture', 'data_sha256':DATA_SHA256,
    'unit':snapshot['series']['display_unit'], 'minimum_observations':10, 'review_threshold':10.0},
    sources=(snapshot['source']['url'],))

@toolkit.tool
def check_finding(values: list[float], difference: float, *, context: toolkit.Context) -> dict:
    """Recompute the comparison and apply the explicitly declared screening threshold."""
    if len(values)<4 or len(values)%2:
        raise ValueError('Invalid comparison windows')
    n=len(values)//2
    exact=mean(values[n:])-mean(values[:n])
    if not math.isclose(difference, exact, rel_tol=1e-10, abs_tol=1e-10):
        raise ValueError('Finding differs from independently recomputed measurement')
    candidate=len(values)>=context.data['minimum_observations'] and abs(exact)>=context.data['review_threshold']
    return {'review': {'verdict':'candidate' if candidate else 'no_candidate',
                       'threshold':context.data['review_threshold'], 'previous_episodes':len(context.episodes),
                       'scope':'Descriptive screening only; no statistical significance or causal claim.'}}

@toolkit.tool
def render_report(measurement: dict, review: dict, years: list[int]) -> dict:
    return {'report':f"{years[0]}–{years[-1]}: difference={measurement['value']:+.3f} index points; "
            f"review={review['verdict']}; threshold={review['threshold']}. {review['scope']}"}

verification_skill = toolkit.skill(name='finding_verification', tools=(check_finding,), instructions='Verify the exact recorded measurement.')
reporting_skill = toolkit.skill(name='evidence_reporting', tools=(render_report,), instructions='Report only recorded measurements and review outcomes.')

# This function declares the complete five-agent / two-system application visibly.
# Every call creates fresh Memory, so profiles remain independent.
def build_application(runtime, framework, label):
    local = runtime.provider == 'python-runtime'
    investigator_skill = toolkit.skill(name='explicit_comparison', tools=(compare_windows,)) if local else exploration_skill
    investigator = toolkit.Agent('investigator', runtime=runtime, framework=framework, skills=(investigator_skill,),
        instructions=(
            'Choose exactly one Tool appropriate to the question. Supply every observation unchanged. '
            'Use profile_series to describe, compare_windows to compare, measure_trend for endpoint slope. '
            'After the Tool completes, report its result without calling another Tool.'))
    challenger = toolkit.Agent('challenger', runtime=runtime, framework=framework, skills=(verification_skill,),
        instructions=(
            'Call check_finding exactly once with the exact supplied arguments. '
            'After the Tool completes, report its result without calling another Tool.'))

    async def explore(state):
        arguments={'values':state.values}
        request={'tool':'compare_windows','input':arguments} if local else state.question+'\nArguments: '+json.dumps(arguments)
        observed=await investigator.arun(request)
        # Preserve failures as RunResult, rather than fabricate a state update.
        if not observed.ok:
            return observed
        calls=[e for e in tool_events(observed) if e.name in {'profile_series','compare_windows','measure_trend'}]
        if len(calls)!=1 or not calls[0].ok or calls[0].input != arguments:
            raise ValueError('Expected one successful Tool with unchanged observations')
        if calls[0].name != 'compare_windows':
            raise ValueError('This investigation requires comparison; inappropriate model selection')
        return {'measurement':business_output(calls[0])['measurement'], 'selected_tool':calls[0].name}

    async def challenge(state):
        arguments={'values':state.values, 'difference':state.measurement['value']}
        request={'tool':'check_finding','input':arguments} if local else 'Call check_finding with these exact arguments: '+json.dumps(arguments)
        observed=await challenger.arun(request)
        if not observed.ok:
            return observed
        calls=[e for e in tool_events(observed) if e.name=='check_finding']
        if len(calls)!=1 or not calls[0].ok or calls[0].input!=arguments:
            raise ValueError('Review must verify exact observations and measurement')
        return {'review':business_output(calls[0])['review']}

    profile_graph=toolkit.Graph(Investigation)
    profile_graph.add_node('profile', profile_series)
    profile_graph.add_edge(toolkit.START,'profile'); profile_graph.add_edge('profile',toolkit.END)
    profiler=toolkit.Agent('profiler',graph=profile_graph,skills=(quality_skill,))
    analysis_graph=toolkit.Graph(Investigation)
    analysis_graph.add_node('validate',validate_measurement)
    analysis_graph.add_edge(toolkit.START,'validate'); analysis_graph.add_edge('validate',toolkit.END)
    analyst=toolkit.Agent('analyst',graph=analysis_graph)
    report_graph=toolkit.Graph(Investigation)
    report_graph.add_node('render',render_report)
    report_graph.add_edge(toolkit.START,'render'); report_graph.add_edge('render',toolkit.END)
    reporter=toolkit.Agent('reporter',graph=report_graph,skills=(reporting_skill,))

    discovery_graph=toolkit.Graph(Investigation)
    discovery_graph.add_node('profile',profiler); discovery_graph.add_node('explore',explore); discovery_graph.add_node('validate',analyst)
    discovery_graph.add_edge(toolkit.START,'profile'); discovery_graph.add_edge('profile','explore')
    discovery_graph.add_edge('explore','validate'); discovery_graph.add_edge('validate',toolkit.END)
    discovery=toolkit.System('discovery_'+label, graph=discovery_graph, context=context, memory=toolkit.Memory(capacity=10))

    review_graph=toolkit.Graph(Investigation)
    review_graph.add_node('challenge',challenge); review_graph.add_node('report',reporter)
    review_graph.add_edge(toolkit.START,'challenge'); review_graph.add_edge('challenge','report'); review_graph.add_edge('report',toolkit.END)
    review=toolkit.System('review_'+label, graph=review_graph, context=context, memory=toolkit.Memory(capacity=10))
    return {'discovery':discovery, 'review':review, 'investigator':investigator, 'challenger':challenger,
            'profiler':profiler,'analyst':analyst,'reporter':reporter}

local_app=build_application(local_runtime,local_framework,'python_native')
discovered=require_ok(save_result('03_discovery',await local_app['discovery'].arun(sample)))
print('Selected:',discovered.data['selected_tool'], 'Measurement checked:',discovered.data['analysis_checked'])

## 4. Review and report with evidence

**user:** Have another capability review the finding before producing the report.

**agentic-systems:** The next cells compute and display the observed response. No model result is pre-recorded or simulated.

**internals:** The second System recomputes the difference from original values and assembles a report. Model agreement is not the acceptance criterion.

**grammar:** System composition → nested Agent evidence → lineage()

**route:** Discovery System → Review System → Challenger → Reporter

In [ ]:
def compose_episode(app):
    graph=toolkit.Graph(Investigation)
    graph.add_node('discover',app['discovery']); graph.add_node('review',app['review'])
    graph.add_edge(toolkit.START,'discover'); graph.add_edge('discover','review'); graph.add_edge('review',toolkit.END)
    return graph

# A coordination System exposes both Systems for a single-record run.
# Environment below reuses the two-System graph directly.
coordinated=toolkit.System('single_investigation',graph=compose_episode(local_app),context=context)
reviewed=require_ok(save_result('04_reviewed',await coordinated.arun(sample)))
display(Markdown('**agentic-systems:** '+reviewed.data['report']))
display(pd.DataFrame([{'execution_id':r.execution_id,'parent':r.parent_execution_id,'status':r.status}
                      for r in reviewed.walk()]))
print(reviewed.human_results())

## 5. Successive real periods and bounded history

**user:** Investigate successive periods, retain prior observations and stop if a candidate requires review.

**agentic-systems:** The next cells compute and display the observed response. No model result is pre-recorded or simulated.

**internals:** Process chronological non-overlapping ten-year segments. Every episode compares its two halves. Stop at the first threshold crossing or the record limit; neither a crossing nor its position is assumed.

**grammar:** Environment → records → accumulated state → update/stop → Memory

**route:** record → Discovery/Review episode → update processed/candidate → stop or next record

In [ ]:
# Anchor ten-year blocks at the first available year; leave the final incomplete block unused.
records=tuple(make_record(first,first+9) for first in range(int(frame.year.min()),int(frame.year.max())-8,10))

def update_round(state, record, result):
    return {'processed':state['processed']+1, 'candidate_found':result.data['review']['verdict']=='candidate'}

def stop_round(state, result):
    return state['candidate_found']

def make_environment(app,label):
    return toolkit.Environment('time_series_lab_'+label,graph=compose_episode(app),records=records,
        state={'processed':0,'candidate_found':False}, update=update_round,stop=stop_round,
        max_steps=len(records),context=context,memory=toolkit.Memory(capacity=10))

local_env=make_environment(local_app,'python_native')
env_result=require_ok(save_result('05_environment',await local_env.arun(timeout_s=180)))
display(pd.DataFrame([{'start':out['years'][0],'end':out['years'][-1],
                      'difference':out['measurement']['value'],'verdict':out['review']['verdict'],
                      'prior_episodes_available':out['review']['previous_episodes']}
                     for out in env_result.data['outputs']]))
print('Environment summary:',env_result.data['summary'])

## 6. Independent result and trajectory acceptance

**user:** Verify the finding and confirm that the required operations used the original observations.

**agentic-systems:** The next cells compute and display the observed response. No model result is pre-recorded or simulated.

**internals:** The numerical oracle uses math.fsum rather than the Tool. Trajectory checks enforce exact arguments, counts and operation order. Separate Evals cover the Agent, System and Environment, wrong expectations and invalid input.

**grammar:** Eval(unit=Agent/System/Environment) → cases → result/trajectory checks → negative controls

**route:** case → independent execution → RunResult → acceptance checks → saved Eval report

In [ ]:
def oracle(values):
    n=len(values)//2
    return math.fsum(values[n:])/n-math.fsum(values[:n])/n

def result_check(result, case):
    m=result.data.get('measurement',{})
    return {'ok': bool(result.data.get('analysis_checked')) and m.get('kind')=='compare'
            and math.isclose(m.get('value',float('nan')),oracle(case['input']['values']),rel_tol=1e-10,abs_tol=1e-10)}

def trajectory_check(result, case):
    relevant=[e for e in tool_events(result) if e.name in {'profile_series','compare_windows','validate_measurement'}]
    return {'ok': [e.name for e in relevant]==['profile_series','compare_windows','validate_measurement']
            and all(e.ok for e in relevant)
            and all(e.input.get('values')==case['input']['values'] for e in relevant)}

def agent_check(result, case):
    events=tool_events(result)
    return {'ok':len(events)==1 and events[0].name=='compare_windows' and events[0].ok
            and events[0].input=={'values':sample['values']}
            and math.isclose(business_output(events[0])['measurement']['value'],oracle(sample['values']),abs_tol=1e-10)}

def environment_check(result, case):
    expected=[]
    for record in records:
        delta=oracle(record['values'])
        expected.append(delta)
        if abs(delta)>=context.data['review_threshold']:
            break
    outputs=result.data.get('outputs',[])
    return {'ok':len(outputs)==len(expected)
        and result.data['summary']['state']['processed']==len(expected)
        and all(math.isclose(out['measurement']['value'],v,abs_tol=1e-10) for out,v in zip(outputs,expected))
        and all(out['values']==record['values'] for out,record in zip(outputs,records))}

positive_case={'input':sample,'expected':{'ok':True}}
agent_eval=toolkit.Eval('actor_acceptance',unit=local_app['investigator'],
    cases=({'input':{'tool':'compare_windows','input':{'values':sample['values']}},'expected':{'ok':True}},),checks=(agent_check,))
system_eval=toolkit.Eval('discovery_acceptance',unit=local_app['discovery'],cases=(positive_case,),checks=(result_check,trajectory_check))
# Fresh application/environment makes the Environment acceptance scenario independent.
env_eval=toolkit.Eval('environment_acceptance',unit=make_environment(build_application(local_runtime,local_framework,'eval'),'eval'),
    cases=({'input':{},'expected':{'ok':True}},),checks=(environment_check,))
eval_summary=[]
for name,evaluation in [('agent',agent_eval),('system',system_eval),('environment',env_eval)]:
    observed=save_result('06_eval_'+name,await evaluation.arun(timeout_s=180))
    require_ok(observed)
    eval_summary.append({'case':name,'expected_acceptance':True,'actual':observed.ok,'test':'PASS'})

def wrong_result_check(result, case):
    return {'ok':math.isclose(result.data['measurement']['value'],oracle(case['input']['values'])+1.0,abs_tol=1e-10)}
wrong_case=positive_case
negative=save_result('06_negative_wrong_answer',await toolkit.Eval('wrong_answer',unit=local_app['discovery'],
    cases=(wrong_case,),checks=(wrong_result_check,trajectory_check)).arun())
assert not negative.ok, 'Wrong expected measurement must be rejected'
eval_summary.append({'case':'wrong expected answer','expected_acceptance':False,'actual':negative.ok,'test':'PASS'})

invalid={**sample,'years':sample['years'][:-1],'values':sample['values'][:-1]}
def failure_check(result,case):
    return {'ok':not result.ok and any(e.name=='compare_windows' and not e.ok for e in tool_events(result))}
invalid_eval=save_result('06_invalid_windows',await toolkit.Eval('odd_windows',unit=comparison_actuator,
    cases=({'input':invalid,'expected':{'ok':False}},),checks=(failure_check,)).arun())
require_ok(invalid_eval)
eval_summary.append({'case':'odd windows rejected','expected_acceptance':True,'actual':invalid_eval.ok,'test':'PASS'})
display(pd.DataFrame(eval_summary))

## ADA session and persistent transport budget

The same borrowed Boto3 session selects IAM. It stays outside serialized runtime configuration, Context, state and Memory. Each destination process must supply its own session.

SQLite retains all historical attempts. A 60 → 100 cap increase is explicit, transactional and audited; model, region, token limit, SDK and dataset cannot change within this ledger. Concurrent sends are counted before transmission. Remaining budget is checked before each live profile and stage; actual sends still enforce the cap. Never change working directory or campaign ID to avoid an exhausted ledger.

A missing ledger stops the live campaign unless `ALLOW_NEW_CAMPAIGN=True` is explicitly set for a first run. An exhausted or insufficient ledger is reported as **BLOCKED_BUDGET**, not model-selection failure.


In [ ]:
# Infrastructure support: does not perform inference when RUN_BEDROCK=False.
import sqlite3, threading

class AttemptBudget:
    def __init__(self, path, config, *, allow_increase_from=None, allow_new=False):
        path = Path(path)
        if not path.exists() and not allow_new:
            raise RuntimeError(
                f'Existing campaign ledger not found: {path}. Use the SAME working directory as before. '
                'For a first-ever campaign only, set ALLOW_NEW_CAMPAIGN=True.')
        self.path = str(path)
        self.config = dict(config)
        with sqlite3.connect(self.path, timeout=10) as db:
            db.execute('BEGIN IMMEDIATE')
            db.execute('CREATE TABLE IF NOT EXISTS campaign (id INTEGER PRIMARY KEY, config TEXT NOT NULL)')
            db.execute('CREATE TABLE IF NOT EXISTS attempts (id INTEGER PRIMARY KEY, timestamp TEXT NOT NULL)')
            db.execute('CREATE TABLE IF NOT EXISTS cap_changes (id INTEGER PRIMARY KEY, timestamp TEXT NOT NULL, old_cap INTEGER, new_cap INTEGER NOT NULL, attempts_at_change INTEGER NOT NULL)')
            existing = db.execute('SELECT config FROM campaign WHERE id=1').fetchone()
            count = db.execute('SELECT COUNT(*) FROM attempts').fetchone()[0]
            if existing:
                previous = json.loads(existing[0])
                old_cap = previous['max_attempts']
                new_cap = config['max_attempts']
                if {k:v for k,v in previous.items() if k!='max_attempts'} != {k:v for k,v in config.items() if k!='max_attempts'}:
                    raise ValueError('Existing campaign model/data/runtime configuration differs; no mutation performed.')
                if new_cap != old_cap:
                    if old_cap != allow_increase_from or new_cap <= old_cap:
                        raise ValueError('Cap change requires an explicit permitted prior cap and a strictly larger new cap.')
                    db.execute('INSERT INTO cap_changes(timestamp,old_cap,new_cap,attempts_at_change) VALUES (?,?,?,?)',
                               (datetime.now(timezone.utc).isoformat(), old_cap, new_cap, count))
                    db.execute('UPDATE campaign SET config=? WHERE id=1', (json.dumps(config,sort_keys=True),))
                    print(f'Explicit cumulative cap increase: {old_cap} -> {new_cap}; retained attempts: {count}')
            else:
                db.execute('INSERT INTO campaign VALUES (1,?)', (json.dumps(config,sort_keys=True),))
                db.execute('INSERT INTO cap_changes(timestamp,old_cap,new_cap,attempts_at_change) VALUES (?,?,?,?)',
                           (datetime.now(timezone.utc).isoformat(), None, config['max_attempts'], count))

    def before_send(self, **kwargs):
        # Never log request, headers, body, URL or credentials.
        with sqlite3.connect(self.path, timeout=10) as db:
            db.execute('BEGIN IMMEDIATE')
            config = json.loads(db.execute('SELECT config FROM campaign WHERE id=1').fetchone()[0])
            count = db.execute('SELECT COUNT(*) FROM attempts').fetchone()[0]
            if count >= config['max_attempts']:
                raise RuntimeError('Persistent Bedrock HTTP attempt cap exhausted')
            db.execute('INSERT INTO attempts(timestamp) VALUES (?)', (datetime.now(timezone.utc).isoformat(),))

    def count(self):
        return self.snapshot()['used']

    def snapshot(self):
        with sqlite3.connect(self.path, timeout=10) as db:
            db.execute('BEGIN')
            config = json.loads(db.execute('SELECT config FROM campaign WHERE id=1').fetchone()[0])
            count = db.execute('SELECT COUNT(*) FROM attempts').fetchone()[0]
            changes = [dict(zip(('timestamp','old_cap','new_cap','attempts_at_change'), row))
                       for row in db.execute('SELECT timestamp,old_cap,new_cap,attempts_at_change FROM cap_changes ORDER BY id')]
        return {'used':count, 'cap':config['max_attempts'], 'remaining':max(0,config['max_attempts']-count), 'cap_changes':changes}

    def require_remaining(self, needed):
        status = self.snapshot()
        if status['remaining'] < needed:
            raise RuntimeError(f"BLOCKED_BUDGET: need at least {needed} sends; remaining={status['remaining']}, used={status['used']}, cumulative cap={status['cap']}. No ledger reset.")
        return status

bedrock_session=None
budget=None
if RUN_BEDROCK:
    import boto3
    from botocore.config import Config
    if not BEDROCK_MODEL_ID.strip():
        raise ValueError('Set the exact BEDROCK_MODEL_ID available in ADA before enabling inference.')
    candidate=ADA_SESSION
    if candidate is not None and hasattr(candidate,'boto_session'):
        candidate=candidate.boto_session
    if candidate is None:
        candidate=boto3.Session(region_name=AWS_REGION)
    if not isinstance(candidate,boto3.session.Session):
        raise TypeError('ADA_SESSION must be a Boto3 Session or a SageMaker Session exposing boto_session.')
    if candidate.region_name != AWS_REGION:
        raise ValueError('Session region and configured region must agree; supply a regional session.')
    bedrock_session=candidate
    campaign={'model':BEDROCK_MODEL_ID,'region':AWS_REGION,'max_attempts':MAX_BEDROCK_HTTP_ATTEMPTS,
              'max_output_tokens':MAX_OUTPUT_TOKENS,'data_sha256':DATA_SHA256,'sdk_version':'2.1.9'}
    budget=AttemptBudget(ROOT/(CAMPAIGN_ID+'.sqlite'), campaign,
                         allow_increase_from=BUDGET_CAP_INCREASE_FROM, allow_new=ALLOW_NEW_CAMPAIGN)
    # Register once per session. New clients copy these hooks. Retain hooks until workers finish.
    hook_id='ts-discovery-218-attempt-budget'
    bedrock_session.events.unregister('before-send.bedrock-runtime',unique_id=hook_id)
    bedrock_session.events.register('before-send.bedrock-runtime',budget.before_send,unique_id=hook_id)
    print('Borrowed Boto3 session; IAM selected; budget:', budget.snapshot())
    (OUTPUT/'budget.json').write_text(json.dumps(budget.snapshot(), indent=2))
else:
    print('Bedrock session/inference: NOT RUN (local profile).')

## 7. Run only the selected Bedrock framework

**user:** Run the pending OpenAI Agents profile using the same ADA session and model.

**agentic-systems:** Outputs below are actual run results. No recorded model answer is substituted.

**internals:** The explicit `LIVE_FRAMEWORKS` tuple selects profiles; it does not automatically rerun Native. Each profile gets fresh Memory. Preflight estimates sends from the independent expected stop rule (two sends per actor). There is no automatic retry of failed stages. All saved evidence and the bundle survive a failed stage.

**grammar:** Runtime + Framework → actor Agents → two Systems → Environment + Eval → evidence

**route:** selected framework → description → Environment → Discovery Eval → summary and bundle

Native uses `max_turns=1`; OpenAI Agents uses `max_turns=2` for tool selection followed by a final answer. Each actor still permits one Tool invocation. Actual HTTP sends, including failures/retries, are the authoritative budget count.


In [ ]:
live_summary=[]
if RUN_BEDROCK:
    expected_episodes = 0
    for record in records:
        expected_episodes += 1
        if abs(oracle(record['values'])) >= context.data['review_threshold']:
            break
    # Description: 2 sends; Environment: 2 actors x 2 sends per episode; Eval: 2 sends.
    planned_sends = 2 + 4 * expected_episodes + 2
    for framework_name in LIVE_FRAMEWORKS:
        label='bedrock_'+framework_name.replace('-','_')
        stage='preflight'
        row={'profile':label, 'status':'RUNNING', 'description_selection':'NOT RUN',
             'environment':'NOT RUN', 'environment_check':None, 'discovery_eval':'NOT RUN'}
        before = budget.count()
        try:
            budget.require_remaining(planned_sends)
            runtime=toolkit.runtime(provider='bedrock-runtime', model=BEDROCK_MODEL_ID, session=bedrock_session,
                max_turns=2 if framework_name=='openai-agents' else 1,
                max_tool_calls=1, max_tokens=MAX_OUTPUT_TOKENS, max_retries=0, timeout_s=120)
            assert runtime.session is bedrock_session
            assert 'session' not in runtime.model_dump()
            (OUTPUT/(label+'.runtime.json')).write_text(runtime.model_dump_json(indent=2))
            app=build_application(runtime,toolkit.framework(framework_name),label)

            stage='description'
            budget.require_remaining(2)
            description_result=save_result(label+'_description',await app['investigator'].arun(
                'Describe these observations; do not compare periods. Arguments: '+json.dumps({'values':sample['values']}),timeout_s=150))
            require_ok(description_result)  # Surface operational failure before selection checks.
            calls=tool_events(description_result)
            description_ok=(len(calls)==1 and calls[0].ok and calls[0].name=='profile_series'
                            and calls[0].input=={'values':sample['values']})
            row['description_selection']='PASS' if description_ok else 'FAIL'
            if not description_ok:
                raise AssertionError('Completed actor did not execute exactly one profile_series Tool with original values.')
            print('Description Tool output:',business_output(calls[0]))

            stage='environment'
            budget.require_remaining(4 * expected_episodes)
            environment_result=save_result(label+'_environment',await make_environment(app,label).arun(timeout_s=600))
            row['environment']=environment_result.status
            require_ok(environment_result)
            row['environment_check']=environment_check(environment_result,{})['ok']
            assert row['environment_check'], 'Environment result/stop check failed; evidence saved.'

            stage='discovery_eval'
            budget.require_remaining(2)
            evaluation=toolkit.Eval('live_discovery_'+label,unit=app['discovery'],cases=(positive_case,),checks=(result_check,trajectory_check))
            evaluated=save_result(label+'_eval',await evaluation.arun(timeout_s=300))
            row['discovery_eval']=evaluated.status
            require_ok(evaluated)
            row['status']='PASS'
        except Exception as exc:
            budget_state=budget.snapshot()
            row['status']='BLOCKED_BUDGET' if ('BLOCKED_BUDGET' in str(exc) or budget_state['remaining']==0) else 'FAIL'
            row['failed_stage']=stage
            row['error_type']=type(exc).__name__
            # Actual RunResult errors were saved before the assertion; do not replace them.
            print(f"{label}: {row['status']} at {stage}; see evidence bundle below.")
            raise
        finally:
            row['http_attempts_reserved']=budget.count()
            row['http_attempts_this_profile']=budget.count()-before
            live_summary.append(row)
            (OUTPUT/'live_summary.json').write_text(json.dumps(live_summary,indent=2))
            (OUTPUT/'budget.json').write_text(json.dumps(budget.snapshot(),indent=2))
            display(pd.DataFrame(live_summary))
            write_run_bundle()
else:
    live_summary=[{'profile':'bedrock/'+name,'status':'NOT RUN'} for name in LIVE_FRAMEWORKS]
    display(pd.DataFrame(live_summary))


## Evidence for 2.1.9 development

The next cell summarizes this execution, without claiming full SDK certification. A local PASS establishes orchestration, arithmetic, checks and evidence behavior for this example. It does not establish model selection, IAM access or cross-framework behavior. Live profiles must be inspected independently.

For acceptance, install the exact 2.1.9 candidate wheel and compare results, Tool arguments, episode counts, failed cases, lineage invariants and request usage against the preserved 2.1.8 baseline. Do not relabel baseline outputs as candidate acceptance.

Publish the executed `.ipynb` and the JSON alongside the example source so GitHub can render actual outputs. Review exported evidence before publishing. Session objects are process-local and are never exported. Do not commit the local request ledger as reusable application configuration.

In [ ]:
summary={'scope':'Time Series Discovery example only','sdk_version':version('agentic-systems'),
    'data_sha256':DATA_SHA256,'local_evaluations':eval_summary,'live':live_summary,
    'result_count':len(RESULTS),'invariants':{k:v.check_invariants().ok for k,v in RESULTS.items()},
    'http_attempts_reserved':budget.count() if budget else 0}
(OUTPUT/'summary.json').write_text(json.dumps(summary,indent=2))
display(Markdown('**Execution evidence saved:** `'+str(OUTPUT)+'`'))
display(pd.DataFrame([{'execution':k,'status':v.status,'invariants':v.check_invariants().ok}
                      for k,v in RESULTS.items()]))
# Deliberate rejecting controls remain failed RunResults and are expected in this table.
assert all(summary['invariants'].values())
write_run_bundle()
